# MiniMax H3 — Colab生成ノートブック（窓際族物語 / colab-video スキル・製品生成専用）

動画スキルで制作済みのバンドル（キーフレーム＋wav＋`ch*_workflow.json`＋`h3_run.py`）を、
ColabのL4 GPUでチャプター毎に動画化する。配管検証は2026-08に完了済みのため、このノートブックは生成だけに特化している。

使い方: **セル1だけ編集**して、あとは上から順に実行（1→8）。パイロット（セリフ有りチャプター）を先に1本生成して確認してから残りを回すこと。
セル9（アドホック生成）は、チャプター定義に縛られず素材＋プロンプトから単発で1本作るときに使う。

課金の注意: CUは「GPUランタイム接続中の時間」で消費される（セル実行中でなくても）。終わったら必ず「ランタイム → ランタイムを接続解除して削除」。セル1の`AUTO_DISCONNECT = True`にすると、セル7の全チャプター完了後に自動で削除される（Drive退避運用時のみ）。


In [ ]:
#@title 0.（初回のみ・無料CPUランタイムでOK）重みをDriveへ事前配置 — GPUセッションのCU消費とDL待ちをなくす
# 使い方: 「ランタイム → ランタイムのタイプを変更 → CPU」にして、このセルだけを実行する（セル1以降は不要）。
# Driveに完全な重みが揃っていれば何もしないので、再実行は常に安全。完了後はこのCPUランタイムを削除してよい。
# 以後のGPUセッションは、セル3がDrive→ローカルのコピーだけで立ち上がる（HFからの大容量DLが消える）。
DRIVE_DIR = "/content/drive/MyDrive/h3_weights"  # GPUセッションのWEIGHTS_DRIVE_DIRと同じ場所
TARGET_GPUS = ["L4"]     # 事前配置する対象GPU。["L4"] / ["A100"] / ["L4", "A100"]（両方＝unet 4本で約117GB。Drive容量に注意）
INCLUDE_I2V = True       # I2Vチャプター用unet（fl2va）も配置する
INCLUDE_R2V = True       # R2Vチャプター用unet（ref2va）も配置する

import os, shutil, subprocess
REPO = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
SIZES = {  # HF上の正確なバイト数（2026-08時点）。MIN_BYTESより厳密な完全性チェックに使う
    "vae/minimax_h3_audio_vae_fp32.safetensors": 605_254_808,
    "vae/minimax_h3_video_vae_fp16.safetensors": 5_207_808_496,
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 27_141_342_152,
    "diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20_958_205_608,
    "diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20_958_205_608,
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20_970_379_616,
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20_970_379_616,
}
UNET_Q = {"L4": "fp8_scaled", "A100": "int8_convrot"}  # Ada=fp8 / Ampere=int8（エンコーダはどちらもint8_convrot）

need = ["vae/minimax_h3_audio_vae_fp32.safetensors",
        "vae/minimax_h3_video_vae_fp16.safetensors",
        "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"]
for g in TARGET_GPUS:
    q = UNET_Q[g]
    if INCLUDE_I2V:
        need.append(f"diffusion_models/minimax_h3_fl2va_pruned_{q}.safetensors")
    if INCLUDE_R2V:
        need.append(f"diffusion_models/minimax_h3_ref2va_pruned_{q}.safetensors")
need = list(dict.fromkeys(need))

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
if not shutil.which("aria2c"):
    subprocess.run(["apt-get", "-qq", "-y", "install", "aria2"], check=True, capture_output=True)

todo = []
for rel in need:
    n = os.path.basename(rel)
    dst = f"{DRIVE_DIR}/{n}"
    if os.path.exists(dst) and os.path.getsize(dst) == SIZES[rel]:
        print("OK（配置済み）", n)
    else:
        todo.append(rel)
total = sum(SIZES[r] for r in todo)
free = shutil.disk_usage(DRIVE_DIR).free
print(f"これから配置: {len(todo)}本 / {total/2**30:.1f} GiB（Drive空き {free/2**30:.1f} GiB）")
assert free >= total + 2_000_000_000, (
    "Drive空き不足。プランを上げるか、不要ファイル削除＋ゴミ箱を空にする（ゴミ箱の中身も容量にカウントされる）")

TMP = "/content/h3_dl"
os.makedirs(TMP, exist_ok=True)
for rel in todo:
    n = os.path.basename(rel)
    local, dst = f"{TMP}/{n}", f"{DRIVE_DIR}/{n}"
    print(f"=== {n} をDL（aria2・16並列・中断してもセル再実行でレジューム） ===", flush=True)
    r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                        "--summary-interval=30", "--console-log-level=warn",
                        "-d", TMP, "-o", n, f"{REPO}/{rel}"])
    assert r.returncode == 0 and os.path.getsize(local) == SIZES[rel], f"{n} のDL不完全 — このセルを再実行"
    print("  -> Driveへコピー中（FUSE越しで数分〜十数分）", flush=True)
    if os.path.exists(dst):
        os.remove(dst)
    shutil.copy(local, dst)
    assert os.path.getsize(dst) == SIZES[rel], f"{n} のDriveコピー不完全 — このセルを再実行"
    os.remove(local)  # ローカルは都度消してディスクを使い回す
    print("  OK", n)
drive.flush_and_unmount()  # キャッシュをDriveへ書き切ってから終了（これが済むまでランタイムを消さない）
print("完了。全重みをDriveへ配置済み。このランタイムは削除してよい")

In [ ]:
#@title 1. 設定＋環境チェック（毎セッションここだけ編集。実行すると選択中のGPUと使用重みを表示）
CHAPTERS = []                # 生成するチャプター。空 = バンドル内の全チャプターを番号順に生成。パイロット運用なら ["ch2"] → 合格後 [] （生成済みは自動スキップ）
EXPECTED_GPU = "A100"            # 例 "A100" / "L4"。設定すると、ランタイムの選択がそれと違うときにここで止まる（設定変更漏れの検知）
BUNDLE_ZIP_FROM_DRIVE = ""   # 例 "/content/drive/MyDrive/41_okayaman_bundle.zip"。空ならセル4でブラウザからアップロード
WEIGHTS_DRIVE_DIR = "/content/drive/MyDrive/h3_weights"       # 例 "/content/drive/MyDrive/h3_weights"。Driveを重みキャッシュに使う（DL回避。実行前に必要分をローカルへコピーする）
SAVE_WEIGHTS_TO_DRIVE = True # WEIGHTS_DRIVE_DIR設定時、HFから落とした重みをDriveへ保存する（次回セッションが数分で立ち上がる）
OUT_DRIVE_DIR = "/content/drive/MyDrive/h3_outputs/"           # 例 "/content/drive/MyDrive/h3_outputs"。設定すると各チャプター完了ごとに即Driveへ退避（切断事故に強い）
NEED_I2V = False              # I2Vチャプター（fl2va 21GB）を使う
NEED_R2V = True              # R2Vチャプター（ref2va 21GB）を使う
COMFY_FLAGS = []             # 生成中に CUDA out of memory が出たら ["--lowvram"] にしてセル6から再実行
AUTO_DISCONNECT = False      # Trueにすると、セル7の全チャプター完了後にDriveへ書き切ってからランタイムを自動削除（課金停止。OUT_DRIVE_DIR設定時のみ機能。パイロットはFalseのまま、放置する本番ランでTrueにする）

# --- 以下は自動判定（編集不要）。このランタイムで実際に選択されているGPUを正として重みを決める ---
import shutil, torch, psutil
if torch.cuda.is_available():
    NAME, CAP = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    # 無料CPUランタイム: 生成はできないが、セル2→3で重みをDL→Driveへ配置する用途（0円）に使える
    NAME, CAP, VRAM = "CPU（重み配置専用モード — セル3まで実行、生成セルは不可）", (8, 9), 0.0
    print("⚠ GPUなし: 重みのDrive配置専用モードとして続行（L4/Ada向けバリアントを選択）")
if EXPECTED_GPU:
    assert EXPECTED_GPU.lower() in NAME.lower(), (
        f"意図したGPU「{EXPECTED_GPU}」と実際の割当「{NAME}」が違う！\n"
        f"ランタイム → ランタイムのタイプを変更 → {EXPECTED_GPU} を選んで再接続してから、このセルを再実行")
if torch.cuda.is_available():
    assert CAP >= (8, 0), f"{NAME} は生成に使えない（Turing以下）。L4以上のGPUを選ぶ"
RAM = psutil.virtual_memory().total / 2**30
DISK = shutil.disk_usage("/content").free / 2**30

if CAP >= (10, 0):        # Blackwell
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", True
elif CAP >= (8, 9):       # Ada (L4/RTX40xx) / Hopper
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors", True
else:                     # Ampere (A100)
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors", False
Q = "fp8_scaled" if FP8 else "int8_convrot"
UNET_I2V = f"minimax_h3_fl2va_pruned_{Q}.safetensors"
UNET_R2V = f"minimax_h3_ref2va_pruned_{Q}.safetensors"

print(f"★ このセッションのGPU: {NAME} (SM {CAP[0]}.{CAP[1]})  VRAM {VRAM:.1f} GiB  RAM {RAM:.1f} GiB  空きディスク {DISK:.1f} GiB")
print(f"★ 使用する重み: {UNET_I2V} / {UNET_R2V} / {ENCODER}")
print("★ 設定:", dict(CHAPTERS=CHAPTERS, NEED_I2V=NEED_I2V, NEED_R2V=NEED_R2V,
                     WEIGHTS_DRIVE_DIR=WEIGHTS_DRIVE_DIR or "(未使用)", OUT_DRIVE_DIR=OUT_DRIVE_DIR or "(未使用)"))

# ディスク見積り: 実行時の重みはローカル実体が必須（Drive FUSE越しのsymlink参照は不可・実測）。
# Drive利用時はユニットを1本ずつ入れ替えるので「エンコーダ27＋VAE6＋ユニット21≒54GB」あれば足りる。
# Drive無しで両ユニットをDLする場合は約75GB必要（L4の実測ディスク65GBには収まらない）。
est = 6 + 27 + (21 if WEIGHTS_DRIVE_DIR else 21 * (NEED_I2V + NEED_R2V))
if DISK < est + 5:
    print(f"⚠ 空き{DISK:.0f}GiBに対し必要見積り約{est}GB — "
          + ("不要ファイルの削除を検討" if WEIGHTS_DRIVE_DIR else "WEIGHTS_DRIVE_DIRの利用か、NEEDフラグを片方ずつにすることを推奨"))

In [ ]:
%%bash
# 2. ComfyUIインストール＋aria2導入（2〜3分）
set -e
apt-get -yq install aria2 > /dev/null 2>&1 || true
cd /content
if [ ! -d ComfyUI ]; then
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
fi
cd ComfyUI
pip install -q -r requirements.txt
test -f comfy_extras/nodes_minimax_h3.py && echo "MiniMax H3 nodes: OK" \
  || { echo "ERROR: nodes_minimax_h3.py が無い — ComfyUIが古い"; exit 1; }


In [ ]:
#@title 3. 重み配置（バックグラウンド実行 — 開始したらそのままセル4〜6へ進んでよい。セル7が完了を待つ）
# GPUごとに使うユニット重みが違う（L4/Ada=fp8ペア、A100/Ampere=int8ペア。各21GB×2）。
# エンコーダとVAE（約33GB）は共通。Driveには「共通分＋今のGPUのペア」だけを置く方針で、
# 必要な重みがDriveに無く、代わりに別バリアントがある場合は削除してから新しい方をDL＆保存する
# （共通33GB＋ペア42GB≒75GBで、Google One 100GBに常に収まる）。
import concurrent.futures, os, shutil, subprocess, threading
REPO = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
SUB = lambda n: "vae" if "_vae_" in n else ("text_encoders" if n.startswith("qwen3vl") else "diffusion_models")
ALT_VARIANTS = {  # 同じモデルの別量子化名（GPU切替時にDriveから退避する対象）
    "minimax_h3_fl2va_pruned_fp8_scaled.safetensors": ["minimax_h3_fl2va_pruned_int8_convrot.safetensors"],
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": ["minimax_h3_fl2va_pruned_fp8_scaled.safetensors"],
    "minimax_h3_ref2va_pruned_fp8_scaled.safetensors": ["minimax_h3_ref2va_pruned_int8_convrot.safetensors"],
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": ["minimax_h3_ref2va_pruned_fp8_scaled.safetensors"],
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": ["qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"],
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": ["qwen3vl_32b_minimax_h3_int8_convrot.safetensors"],
}
MIN_BYTES = {  # 不完全ファイル検出用の下限（実サイズの少し下）
    "minimax_h3_video_vae_fp16.safetensors": 5_000_000_000,
    "minimax_h3_audio_vae_fp32.safetensors": 550_000_000,
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 26_000_000_000,
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15_000_000_000,
    "minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20_900_000_000,
    "minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20_900_000_000,
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20_000_000_000,
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20_000_000_000,
}
need = ["minimax_h3_video_vae_fp16.safetensors", "minimax_h3_audio_vae_fp32.safetensors", ENCODER]
if NEED_I2V: need.append(UNET_I2V)
if NEED_R2V: need.append(UNET_R2V)

if WEIGHTS_DRIVE_DIR or BUNDLE_ZIP_FROM_DRIVE or OUT_DRIVE_DIR:
    from google.colab import drive as _gd
    _gd.mount("/content/drive")  # OAuthが出ることがあるのでここだけ前面で実行
drive_dir = WEIGHTS_DRIVE_DIR or None
if drive_dir:
    os.makedirs(drive_dir, exist_ok=True)

def ok_size(path, n):
    return os.path.exists(path) and os.path.getsize(path) >= MIN_BYTES.get(n, 1)

DRIVE_IO_LOCK = threading.Lock()  # キャッシュ掃除のunmountと、他セルのDrive読み出しの衝突防止

def purge_drivefs_cache():
    # DriveFSはFUSE読み出しの内容キャッシュをローカルディスクにも書くため、大物コピー中に
    # 「コピー先＋キャッシュ」の二重消費でディスクが枯渇することがある（ENOSPC実測・2026-08）。
    # unmount→キャッシュ削除→remountで空ける。
    from google.colab import drive as _gd
    lock = globals().get("DRIVE_IO_LOCK")
    if lock:
        lock.acquire()
    try:
        print("  空きディスク逼迫 → DriveFSキャッシュを掃除（unmount→削除→remount・数十秒）", flush=True)
        _gd.flush_and_unmount()
        shutil.rmtree("/root/.config/Google/DriveFS", ignore_errors=True)
        _gd.mount("/content/drive")
    finally:
        if lock:
            lock.release()

def copy_from_drive(src, dst, floor=3 * 2**30, threads=8, chunk=64 * 2**20):
    # Drive→ローカルのレジューム可能コピー。並列pread（8スレッド・実測62→83MB/s）で読み、
    # 追記順を守って書く＝dstのファイルサイズがそのままレジューム点になる。
    # 空きがfloorを切ったらDriveFSキャッシュを掃除して続行する（shutil.copyだとENOSPCで落ちる）。
    size = os.path.getsize(src)
    done = os.path.getsize(dst) if os.path.exists(dst) else 0

    def read_round(start):  # startからthreads*chunk分を並列preadで読んで返す
        n = min(threads * chunk, size - start)
        def one(i):
            off, ln = start + i * chunk, min(chunk, n - i * chunk)
            if ln <= 0:
                return b""
            fd = os.open(src, os.O_RDONLY)
            try:
                parts, got = [], 0
                while got < ln:
                    b = os.pread(fd, ln - got, off + got)
                    assert b, f"{src} の読み出しが途切れた — セルの再実行で続きから再開する"
                    parts.append(b)
                    got += len(b)
                return b"".join(parts)
            finally:
                os.close(fd)
        with concurrent.futures.ThreadPoolExecutor(threads) as ex:
            return b"".join(ex.map(one, range(threads)))

    with concurrent.futures.ThreadPoolExecutor(1) as ahead:
        nxt = None  # 先読み: 書き込みと次ラウンドの読みを重ねる
        while done < size:
            buf = nxt.result() if nxt else None
            nxt = None
            if shutil.disk_usage("/content").free < floor:
                purge_drivefs_cache()
                assert shutil.disk_usage("/content").free >= floor, \
                    "キャッシュ掃除後も空き不足 — 不要ファイルを消して再実行（コピーは途中から再開する）"
            if buf is None:
                buf = read_round(done)
            if done + len(buf) < size:
                nxt = ahead.submit(read_round, done + len(buf))
            with open(dst, "ab") as fo:
                fo.write(buf)
            if done // (4 * 2**30) != (done + len(buf)) // (4 * 2**30):
                print(f"    ... {(done + len(buf)) / 2**30:.0f}/{size / 2**30:.0f} GiB", flush=True)
            done += len(buf)

def _place_weights():
    for n in need:
        dst = f"/content/ComfyUI/models/{SUB(n)}/{n}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if ok_size(dst, n):
            print("skip（配置済み）", n)
            continue
        drv = f"{drive_dir}/{n}" if drive_dir else None
        if drv and ok_size(drv, n):
            # 注意: Drive(FUSE)越しの直接参照symlinkは実行時に使えない（comfy_aimdoの
            # read_file_sliceがFUSE上で失敗する — 2026-08実測）。Driveはダウンロード回避の
            # キャッシュとして使い、実行前に必要な重みをローカルへコピーする。
            if os.path.lexists(dst):
                os.remove(dst)
            if SUB(n) == "diffusion_models":
                # ユニット2本はディスクに同時に載らないことがあるため、セル7がチャプター毎にローカル化する
                os.symlink(drv, dst)
                print("drive OK（実体はセル7で使用直前にローカル化）", n)
            else:
                print(f"Driveからローカル化中（8スレッド並列コピー。中断してもレジューム可）: {n}", flush=True)
                copy_from_drive(drv, dst + ".part")
                os.replace(dst + ".part", dst)
                print("drive OK（ローカル化済み）", n)
            continue
        if drv and SAVE_WEIGHTS_TO_DRIVE:
            # GPU切替: これから保存する分の容量がDriveに足りるなら旧バリアントは保持する（両方あれば
            # 次の切替コストがゼロになる）。足りない場合のみ削除して空ける。
            # 注意: Driveの削除はゴミ箱行きで、ゴミ箱の中身も容量にカウントされる。削除しても保存が
            # 容量不足で失敗する場合は https://drive.google.com でゴミ箱を空にする（保存失敗しても生成は続行される）。
            need_bytes = MIN_BYTES.get(n, 0) + 2_000_000_000
            for alt in ALT_VARIANTS.get(n, []):
                alt_path = f"{drive_dir}/{alt}"
                if not os.path.exists(alt_path):
                    continue
                if shutil.disk_usage(drive_dir).free >= need_bytes:
                    print(f"  Drive容量に余裕があるため別バリアントを保持: {alt}")
                    continue
                os.remove(alt_path)
                print(f"  Drive容量確保のため旧バリアントを削除（ゴミ箱行き）: {alt}")
        print(f"=== {n} をaria2でDL（16並列・15秒毎に進捗表示・中断してもレジューム可） ===", flush=True)
        r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                            "--summary-interval=15", "--console-log-level=warn",
                            "-d", os.path.dirname(dst), "-o", n, f"{REPO}/{SUB(n)}/{n}"])
        assert r.returncode == 0 and ok_size(dst, n), f"{n} のDLに失敗 — このセルを再実行すれば途中から再開する"
        print("hf OK", n)
        if drv and SAVE_WEIGHTS_TO_DRIVE:
            try:
                print(f"  -> Driveへ保存中（FUSE越しで時間がかかる。次回以降の高速化用）: {drv}", flush=True)
                shutil.copy(dst, drv)
            except OSError as e:  # Drive容量・一時キャッシュ枯渇などでも生成は止めない
                print(f"  ⚠ Drive保存に失敗（生成には影響なし。後で無料CPUセッションでの配置を推奨）: {e}")
                if os.path.exists(drv):
                    os.remove(drv)
    print(f"★ 重み配置 完了。空きディスク: {shutil.disk_usage('/content').free / 2**30:.1f} GiB", flush=True)

# 配置はバックグラウンドで実行し、ComfyUIインストール（セル2実行済み）後の
# バンドル投入（セル4）〜ComfyUI起動（セル6）と並行させる。完了はセル7の冒頭で待つ。
if "WEIGHTS_THREAD" in globals() and WEIGHTS_THREAD.is_alive():
    print("前回の重み配置がまだ実行中 — 完了を待ってから続行")
    WEIGHTS_THREAD.join()
WEIGHTS_ERR = []

def _bg_place():
    try:
        _place_weights()
    except BaseException as e:
        WEIGHTS_ERR.append(e)
        print(f"⚠ 重み配置が失敗: {e} — セル3を再実行（コピー/DLは途中から再開する）", flush=True)

def wait_weights():
    if WEIGHTS_THREAD.is_alive():
        print("重み配置（バックグラウンド）の完了を待機中...", flush=True)
    WEIGHTS_THREAD.join()
    assert not WEIGHTS_ERR, f"重み配置が失敗している: {WEIGHTS_ERR[0]} — セル3を再実行"

WEIGHTS_THREAD = threading.Thread(target=_bg_place, daemon=True)
WEIGHTS_THREAD.start()
print("★ 重み配置をバックグラウンドで開始 — このままセル4〜6を進めてOK（セル7の冒頭で完了を待つ）")

In [ ]:
#@title 4. バンドル投入（zipをアップロード or Driveから）→ ComfyUI/input/ へ配備
import glob, os, shutil, threading, zipfile

if BUNDLE_ZIP_FROM_DRIVE:
    assert os.path.exists(BUNDLE_ZIP_FROM_DRIVE), f"{BUNDLE_ZIP_FROM_DRIVE} が無い"
    zp = "/content/" + os.path.basename(BUNDLE_ZIP_FROM_DRIVE)
    # セル3のバックグラウンド重み配置がDriveFSキャッシュ掃除（unmount）をすることがあるため、
    # Driveからの読み出しはロックを取ってローカルへ写してから使う
    with globals().get("DRIVE_IO_LOCK") or threading.Lock():
        shutil.copy(BUNDLE_ZIP_FROM_DRIVE, zp)
else:
    from google.colab import files
    print("バンドルzip（<NN>_<slug>_bundle.zip）を選択:")
    up = files.upload()
    zp = "/content/" + next(iter(up))

shutil.rmtree("/content/bundle", ignore_errors=True)
zipfile.ZipFile(zp).extractall("/content/bundle")
hits = glob.glob("/content/bundle/**/script.md", recursive=True)
assert hits, "zip内にscript.mdが見つからない — ラン専用ディレクトリごとzipしたか確認"
BUNDLE = os.path.dirname(hits[0])
print("BUNDLE =", BUNDLE)

inp = "/content/ComfyUI/input"
os.makedirs(inp, exist_ok=True)
n = 0
for p in sorted(glob.glob(f"{BUNDLE}/*.png") + glob.glob(f"{BUNDLE}/*.wav")):
    if os.path.basename(p).startswith("ref_canvas_"):
        continue
    shutil.copy(p, inp)
    n += 1
print(f"{n} files -> ComfyUI/input/")

In [ ]:
#@title 5. workflowをこのGPUの重み名に調整（SaveVideoのcodec補完込み）
import glob, json, os
wfs = sorted(glob.glob(f"{BUNDLE}/ch*_workflow.json"))
assert wfs, f"{BUNDLE} に ch*_workflow.json が無い — バンドル作成時にworkflowを生成したか確認"
for wf in wfs:
    with open(wf) as f:
        d = json.load(f)
    for node in d.values():
        ins = node.get("inputs", {})
        for k, v in ins.items():
            if not isinstance(v, str):
                continue
            if v.startswith("minimax_h3_fl2va"):
                ins[k] = UNET_I2V
            elif v.startswith("minimax_h3_ref2va"):
                ins[k] = UNET_R2V
            elif v.startswith("qwen3vl_32b"):
                ins[k] = ENCODER
        if node.get("class_type") == "SaveVideo":  # ComfyUI新版(2026-08〜)はcodec必須
            ins.setdefault("codec", "auto")
            ins.setdefault("format", "auto")
    with open(wf, "w") as f:
        json.dump(d, f, indent=1)
    print("adjusted", os.path.basename(wf))
print("重み:", UNET_I2V, "/", UNET_R2V, "/", ENCODER)


In [ ]:
#@title 6. ComfyUI起動（プロセスが落ちたらこのセルを再実行）
import json, subprocess, sys, time, urllib.request
SERVER = "127.0.0.1:8188"
subprocess.run(["pkill", "-f", "main.py --listen"], check=False)
time.sleep(2)
LOG = open("/content/comfyui.log", "w")
PROC = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", *COMFY_FLAGS],
    cwd="/content/ComfyUI", stdout=LOG, stderr=subprocess.STDOUT)
INFO = None
for _ in range(90):
    try:
        INFO = json.load(urllib.request.urlopen(f"http://{SERVER}/object_info", timeout=5))
        break
    except Exception:
        time.sleep(2)
assert INFO, "ComfyUIが起動しない — !tail -50 /content/comfyui.log で確認"
h3_nodes = sorted(k for k in INFO if k.startswith("MiniMaxH3"))
assert h3_nodes, "H3ノードが登録されていない — ComfyUIのバージョンを確認"
print("MiniMaxH3 nodes:", h3_nodes)


In [ ]:
#@title 7. チャプター生成（モード順に並べ替え→完了ごとに即退避。生成済みはスキップ＝中断・再開に強い）
import concurrent.futures, glob, json, os, re, shutil, subprocess, sys
if "wait_weights" in globals():
    wait_weights()  # セル3のバックグラウンド重み配置の完了を待つ
if not CHAPTERS:  # 未指定 = バンドル内の全チャプター
    CHAPTERS = [os.path.basename(w)[: -len("_workflow.json")] for w in glob.glob(f"{BUNDLE}/ch*_workflow.json")]
    print("CHAPTERS未指定 → 全チャプターを生成")
assert CHAPTERS, f"{BUNDLE} に ch*_workflow.json が無い — バンドルを確認"

def chapter_units(ch):
    with open(os.path.join(BUNDLE, f"{ch}_workflow.json")) as f:
        g = json.load(f)
    return sorted({v for node in g.values() for k, v in node.get("inputs", {}).items() if k == "unet_name"})

# 同じユニットを使うチャプターを連続させ、21GBのユニット入れ替え回数を最小化する
# （同一モード内は番号順を維持。生成順が変わるだけで成果物は同じ）
CHAPTERS = sorted(CHAPTERS, key=lambda ch: (chapter_units(ch), int(re.sub(r"\D", "", ch) or 0)))
print("生成順:", CHAPTERS)
os.makedirs("/content/outputs", exist_ok=True)
if OUT_DRIVE_DIR:
    os.makedirs(OUT_DRIVE_DIR, exist_ok=True)
DIFF_DIR = "/content/ComfyUI/models/diffusion_models"

def purge_drivefs_cache():
    # DriveFSはFUSE読み出しの内容キャッシュをローカルディスクにも書くため、大物コピー中に
    # 「コピー先＋キャッシュ」の二重消費でディスクが枯渇することがある（ENOSPC実測・2026-08）。
    # unmount→キャッシュ削除→remountで空ける。
    from google.colab import drive as _gd
    lock = globals().get("DRIVE_IO_LOCK")
    if lock:
        lock.acquire()
    try:
        print("  空きディスク逼迫 → DriveFSキャッシュを掃除（unmount→削除→remount・数十秒）", flush=True)
        _gd.flush_and_unmount()
        shutil.rmtree("/root/.config/Google/DriveFS", ignore_errors=True)
        _gd.mount("/content/drive")
    finally:
        if lock:
            lock.release()

def copy_from_drive(src, dst, floor=3 * 2**30, threads=8, chunk=64 * 2**20):
    # Drive→ローカルのレジューム可能コピー。並列pread（8スレッド・実測62→83MB/s）で読み、
    # 追記順を守って書く＝dstのファイルサイズがそのままレジューム点になる。
    # 空きがfloorを切ったらDriveFSキャッシュを掃除して続行する（shutil.copyだとENOSPCで落ちる）。
    size = os.path.getsize(src)
    done = os.path.getsize(dst) if os.path.exists(dst) else 0

    def read_round(start):  # startからthreads*chunk分を並列preadで読んで返す
        n = min(threads * chunk, size - start)
        def one(i):
            off, ln = start + i * chunk, min(chunk, n - i * chunk)
            if ln <= 0:
                return b""
            fd = os.open(src, os.O_RDONLY)
            try:
                parts, got = [], 0
                while got < ln:
                    b = os.pread(fd, ln - got, off + got)
                    assert b, f"{src} の読み出しが途切れた — セルの再実行で続きから再開する"
                    parts.append(b)
                    got += len(b)
                return b"".join(parts)
            finally:
                os.close(fd)
        with concurrent.futures.ThreadPoolExecutor(threads) as ex:
            return b"".join(ex.map(one, range(threads)))

    with concurrent.futures.ThreadPoolExecutor(1) as ahead:
        nxt = None  # 先読み: 書き込みと次ラウンドの読みを重ねる
        while done < size:
            buf = nxt.result() if nxt else None
            nxt = None
            if shutil.disk_usage("/content").free < floor:
                purge_drivefs_cache()
                assert shutil.disk_usage("/content").free >= floor, \
                    "キャッシュ掃除後も空き不足 — 不要ファイルを消して再実行（コピーは途中から再開する）"
            if buf is None:
                buf = read_round(done)
            if done + len(buf) < size:
                nxt = ahead.submit(read_round, done + len(buf))
            with open(dst, "ab") as fo:
                fo.write(buf)
            if done // (4 * 2**30) != (done + len(buf)) // (4 * 2**30):
                print(f"    ... {(done + len(buf)) / 2**30:.0f}/{size / 2**30:.0f} GiB", flush=True)
            done += len(buf)

def ensure_local_unet(ch):
    # 実行時の重みはローカル実体が必須（Drive FUSE越しのsymlinkはcomfy_aimdoが読めない）。
    # このチャプターが使うunetをDriveからローカル化し、ディスクが足りなければ他のunetの
    # ローカル実体を削除して空ける（Driveに実体があるので消してよい）。
    for u in chapter_units(ch):
        pth = f"{DIFF_DIR}/{u}"
        drv = f"{WEIGHTS_DRIVE_DIR}/{u}" if WEIGHTS_DRIVE_DIR else None
        if os.path.exists(pth) and not os.path.islink(pth):
            if not (drv and os.path.exists(drv)) or os.path.getsize(pth) == os.path.getsize(drv):
                continue  # 完全なローカル実体
            os.rename(pth, pth + ".part")  # 書きかけ（前回のENOSPC等）→ レジュームで続きから
        assert drv and os.path.exists(drv), f"{u} が未配置 — セル3を実行"
        part = pth + ".part"
        need = os.path.getsize(drv) - (os.path.getsize(part) if os.path.exists(part) else 0) + 4 * 2**30
        for other in sorted(glob.glob(f"{DIFF_DIR}/*.safetensors") + glob.glob(f"{DIFF_DIR}/*.part")):
            if other in (pth, part) or os.path.islink(other):
                continue
            if shutil.disk_usage("/content").free >= need:
                break
            os.remove(other)
            print("  ディスク確保のためローカルunetを削除（Driveに実体あり）:", os.path.basename(other))
        if shutil.disk_usage("/content").free < need:
            purge_drivefs_cache()
        assert shutil.disk_usage("/content").free >= need, "ディスク不足 — 不要ファイルを整理してこのセルを再実行"
        print(f"  {u} をDriveからローカル化中（8スレッド並列コピー。中断してもレジューム可）...", flush=True)
        copy_from_drive(drv, part)
        if os.path.lexists(pth):
            os.remove(pth)
        os.replace(part, pth)
        print("  local OK", u)

for ch in CHAPTERS:
    wf = os.path.join(BUNDLE, f"{ch}_workflow.json")
    assert os.path.exists(wf), f"{wf} が無い"
    ensure_local_unet(ch)
    out = f"/content/outputs/{ch}.mp4"
    if os.path.exists(out):
        print("skip（生成済み）", ch)
        continue
    print(f"=== {ch} 生成開始（動画1秒あたり約4.5分@L4。ポーリング出力が続いていれば正常）===", flush=True)
    r = subprocess.run([sys.executable, os.path.join(BUNDLE, "h3_run.py"), wf, "--out", out])
    if r.returncode != 0:
        raise RuntimeError(f"{ch} が失敗 — !tail -80 /content/comfyui.log で確認。成功済み分はセル8で回収できる")
    if OUT_DRIVE_DIR:
        shutil.copy(out, OUT_DRIVE_DIR)
        print(f"  -> Drive退避済み: {OUT_DRIVE_DIR}/{ch}.mp4")
print("指定チャプター完了")
if AUTO_DISCONNECT and OUT_DRIVE_DIR:
    # 生成失敗時はここに来ない（例外で停止）: ログ診断のためセッションは残る
    print("Driveへ書き切ってからランタイムを削除する（課金停止）...", flush=True)
    from google.colab import drive as _gd, runtime as _rt
    with globals().get("DRIVE_IO_LOCK") or __import__("threading").Lock():
        _gd.flush_and_unmount()  # DriveFSキャッシュを書き切る（これを飛ばすと最後のmp4がDriveに残らないことがある）
    _rt.unassign()  # 「ランタイム → ランタイムを接続解除して削除」と同じ。以降のセルは実行できない
elif AUTO_DISCONNECT:
    print("⚠ OUT_DRIVE_DIRが未設定のため自動切断を中止（切断するとローカルのmp4が消える）— セル8で回収してから手動で削除")
else:
    print("接続は継続中（課金も継続）。ブラウザにも落とすならセル8へ。済んだら手動で「ランタイム → ランタイムを接続解除して削除」"
          "（次回からはセル1の AUTO_DISCONNECT = True で自動化できる）")

In [ ]:
#@title 8. 成果物の回収（zip→ブラウザDL。Drive退避済みならスキップ可）
import glob, subprocess
outs = sorted(glob.glob("/content/outputs/*.mp4"))
assert outs, "/content/outputs にmp4が無い"
print(*outs, sep="\n")
subprocess.run(["zip", "-j", "-q", "/content/h3_outputs.zip", *outs], check=True)
from google.colab import files
files.download("/content/h3_outputs.zip")
print("回収したら「ランタイム → ランタイムを接続解除して削除」で課金を止めること")


In [ ]:
#@title 9.（任意）アドホック生成 — 画像・音声・プロンプトを直接指定して1本作る
# チャプター定義に縛られない単発生成。素材はバンドル同梱ファイル名で指定（新素材は左のファイルペインで
# /content/ComfyUI/input/ へドラッグ＆ドロップしてから指定）。framesは17k+5グリッド（90,124,141,158,...）。
# H3の埋め込み音声は入力wavと同等（2026-08実測）なので、出力の音声はそのまま最終成果物に使える。
ADHOC = dict(
    mode="r2v",      # "i2v"=開始/終了フレーム固定・音声なし / "r2v"=参照画像(≦9)+音声(≦3・各2〜15s)・リップシンク
    frames=124,
    prompt="Required attached input files: <Picture 1> = XXX.png — ...; <Audio 1> = YYY.wav — spoken line, use AS-IS. "
           "The video starts EXACTLY on <Picture 1>. ... (S1) speaks — he says <d>[Japanese] セリフ</d>, lip-syncing to <Audio 1>. "
           "Soundscape: ... Music: no background music.",
    first="chN_start.png", last="chN_end.png",  # i2vのみ
    images=["XXX.png"], audio=["YYY.wav"],      # r2vのみ（<Picture N>/<Audio N>の接続順）
    out="adhoc1",
)
import json, os, subprocess, sys
pf = os.path.join(BUNDLE, f"{ADHOC['out']}_prompt.txt")
with open(pf, "w") as f:
    f.write(ADHOC["prompt"])
wf = os.path.join(BUNDLE, f"{ADHOC['out']}_workflow.json")
cmd = [sys.executable, os.path.join(BUNDLE, "build_h3_workflow.py"), "--mode", ADHOC["mode"],
       "--out", wf, "--prompt-file", pf, "--frames", str(ADHOC["frames"]),
       "--prefix", f"video/{ADHOC['out']}",
       "--encoder", ENCODER, "--unet-i2v", UNET_I2V, "--unet-r2v", UNET_R2V]
if ADHOC["mode"] == "i2v":
    cmd += ["--first", ADHOC["first"], "--last", ADHOC["last"]]
else:
    for im in ADHOC["images"]:
        cmd += ["--image", im]
    for au in ADHOC["audio"]:
        cmd += ["--audio", au]
subprocess.run(cmd, check=True)
with open(wf) as f:  # 旧builder対策のcodec補完
    _d = json.load(f)
for _n in _d.values():
    if _n.get("class_type") == "SaveVideo":
        _n["inputs"].setdefault("codec", "auto")
        _n["inputs"].setdefault("format", "auto")
with open(wf, "w") as f:
    json.dump(_d, f, indent=1)
if "wait_weights" in globals():
    wait_weights()  # セル3のバックグラウンド重み配置の完了を待つ
if "ensure_local_unet" in globals():
    ensure_local_unet(ADHOC["out"])  # 使用unetのローカル実体を確保（セル7と同じ仕組み）
os.makedirs("/content/outputs", exist_ok=True)
r = subprocess.run([sys.executable, os.path.join(BUNDLE, "h3_run.py"), wf,
                    "--out", f"/content/outputs/{ADHOC['out']}.mp4"])
print("結果:", "完了 -> セル8で回収" if r.returncode == 0 else "失敗 — !tail -80 /content/comfyui.log")


## 後工程（ローカル）

回収した`chN.mp4`をラン専用ディレクトリに置き、ffmpegでconcat結合する（埋め込み音声をそのまま使う）。
エンドカード等の文字入れはCapCutで後付け（H3には文字を描かせない）。手順の詳細はランの`H3_COLAB.md`／
`.claude/skills/colab-video/SKILL.md`を参照。

トラブル時はセル出力と `!tail -80 /content/comfyui.log` をClaude Code / Cursorに貼れば診断できる。
